# Virtualizing a collection of h5ad files into one big matrix

You have a folder of `.h5ad` files — one per sample, all sharing the same genes.
Together they're a single dataset of many cells. This notebook references them
into **one logical expression matrix `X`** inside an [Icechunk](https://icechunk.io)
repo **without copying the array bytes**: the repo stores byte-range pointers
into the original files, and a row slice reads only the bytes it needs.

The result is a normal, queryable `AnnData` whose matrix lives, virtually, across
the original files.

> **Scope.** We virtualize the sparse `X` (the bulk of any `.h5ad`). The
> variable-length string columns of `obs`/`var` aren't byte-range addressable, so
> they're loaded from the source files rather than virtualized.

## 1. A collection to work with

We use the classic pbmc3k dataset, sharded along cells into three files — the
same shape as a real per-sample collection (shared genes, disjoint cells). The
cell below fetches and builds it if needed.

In [1]:
import os; os.environ.setdefault('RUST_LOG', 'error')  # quiet icechunk's Rust logs
import os, glob, anndata as ad, numpy as np

os.makedirs('data/collection', exist_ok=True)
if not os.path.exists('data/pbmc3k_modern.h5ad'):
    import urllib.request
    urllib.request.urlretrieve('https://falexwolf.de/data/pbmc3k_raw.h5ad', 'data/pbmc3k_raw.h5ad')
    ad.read_h5ad('data/pbmc3k_raw.h5ad').write_h5ad('data/pbmc3k_modern.h5ad', compression='gzip')
if not glob.glob('data/collection/shard_*.h5ad'):
    a = ad.read_h5ad('data/pbmc3k_modern.h5ad'); b = np.linspace(0, a.n_obs, 4).astype(int)
    for i in range(3):
        a[b[i]:b[i+1]].copy().write_h5ad(f'data/collection/shard_{i}.h5ad', compression='gzip')

paths = [os.path.abspath(p) for p in sorted(glob.glob('data/collection/shard_*.h5ad'))]
for p in paths:
    print(f'{p}  ({os.path.getsize(p)/1e6:.1f} MB)')

/Users/ian/Documents/dev/virtual-h5ad/data/collection/shard_0.h5ad  (4.0 MB)
/Users/ian/Documents/dev/virtual-h5ad/data/collection/shard_1.h5ad  (4.0 MB)
/Users/ian/Documents/dev/virtual-h5ad/data/collection/shard_2.h5ad  (4.0 MB)


## 2. Why the obvious approach fails

A sparse `X` is three 1-D arrays — `data`, `indices`, `indptr` (CSR). Stacking
samples vertically just concatenates `data` and `indices` (column indices don't
change — same genes). The catch is chunking: every file chunks `data`
differently and ends on a *partial* chunk, so a single regular-grid Zarr array
can't span them. VirtualiZarr says so directly:

In [2]:
import xarray as xr
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import LocalStore

DATA = os.path.abspath('data/collection')
reg = ObjectStoreRegistry({f'file://{DATA}': LocalStore(prefix=DATA)})
ds = [open_virtual_dataset(url=f'file://{p}', registry=reg,
                           parser=HDFParser(group='X', drop_variables=['indices', 'indptr']))
      for p in paths]
try:
    xr.concat(ds, dim='phony_dim_0')
except Exception as e:
    print(f'{type(e).__name__}: {e}')

ValueError: Cannot concatenate arrays with inconsistent chunk shapes: (5973,) vs (5939,) .Requires ZEP003 (Variable-length Chunks).


## 3. The fix: rectilinear chunks + native tails

That limit is VirtualiZarr's *regular-grid assumption*, not a real wall. Zarr now
supports **rectilinear (variable-length) chunk grids**, and Icechunk stores chunk
references by coordinate regardless of grid shape. So we build the combined array
ourselves: its grid is every file's HDF5 chunk edges laid end to end, and each
grid cell points at one source chunk's bytes.

One wrinkle: HDF5 pads each array's *final* chunk to full size (e.g. 5939 stored,
5918 valid). That padding can't leak across a file boundary, so each file's tail
chunk is materialized as one small **native** chunk. Everything else stays
virtual. This logic lives in `tutorial/rectilinear_concat.py` (the part
VirtualiZarr doesn't do yet).

In [3]:
import shutil
from tutorial.rectilinear_concat import virtualize_matrix

shutil.rmtree('repo/collection', ignore_errors=True)
snap = virtualize_matrix(paths, 'repo/collection')   # accepts a list, glob, or directory
print('committed snapshot', snap)

  2026-06-17T19:47:01.540254Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196

committed snapshot G77TSEKA5D8XD07B4290


In [4]:
def dir_bytes(p):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(p) for f in fs)

src, repo = dir_bytes('data/collection'), dir_bytes('repo/collection')
print(f'source files : {src/1e6:6.1f} MB')
print(f'icechunk repo: {repo/1e3:6.0f} KB   ({src/repo:.0f}x smaller — references, not copies)')

source files :   12.0 MB
icechunk repo:     66 KB   (183x smaller — references, not copies)


## 4. Open it as one AnnData

No flags or paths to set: the module enables rectilinear chunks on import, and
`open_anndata` self-authorizes from the repo's own config and reattaches
`obs`/`var` from the recorded source files. `X` stays virtual.

In [5]:
from tutorial.rectilinear_concat import open_anndata

A = open_anndata('repo/collection')
A

  2026-06-17T19:47:01.573812Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196

  2026-06-17T19:47:01.574762Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196



AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'

In [6]:
ref = ad.read_h5ad('data/pbmc3k_modern.h5ad')
print('n_obs x n_vars :', A.shape)
print('X identical to original pbmc3k :', (A.X != ref.X).nnz == 0)
print('obs index reattached          :', list(A.obs_names[:3]), '...')

n_obs x n_vars : (2700, 32738)
X identical to original pbmc3k : True
obs index reattached          : ['AAACATACAACCAC-1', 'AAACATTGAGCTAC-1', 'AAACATTGATCAGC-1'] ...


## 5. Reads are lazy

Slicing a row pulls only that row's chunks from the original `.h5ad` files —
nothing was duplicated to build the repo.

In [7]:
import zarr
from tutorial.rectilinear_concat import open_repo

x = zarr.open_group(open_repo('repo/collection').readonly_session('main').store, mode='r')['X']
ip = x['indptr'][:]
r = 1500
s, e = int(ip[r]), int(ip[r + 1])
row = x['data'][s:e]          # fetches just this row's source chunks
print(f'row {r}: {e - s} nonzeros; first values {row[:5]}')
print('grid kind:', x['data'].metadata.chunk_grid.to_dict()['name'])

  2026-06-17T19:47:01.783353Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:196

row 1500: 380 nonzeros; first values [1. 1. 1. 7. 1.]
grid kind: rectilinear


## Next steps

- **Bigger, real collections.** The same `virtualize_matrix(directory, repo)`
  call works on a folder of real per-sample h5ad files (e.g. CELLxGENE / Allen
  Institute immunology datasets) — provided they **share the same gene set**
  (`var`). Files with different gene panels need alignment first, or the simpler
  group-per-sample layout (`experiments/03_virtual_collection.py`).
- **Remote sources.** Point the virtual-chunk container at `s3://…` and the same
  references work over object storage.
- **Caveat.** Rectilinear chunks are experimental in Zarr (flag-gated on read and
  write); the module sets the flag for you.